In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px

In [14]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

grade_mapping = {"F": 0, "D": 1, "C": 2, "B": 3, "A": 4}

skyline["final_grade_numeric"] = skyline["final_grade"].map(grade_mapping)

pearson_r, pearson_p = stats.pearsonr(
    skyline["hours_studied"],
    skyline["final_grade_numeric"],
)

spearman_r, spearman_p = stats.spearmanr(
    skyline["hours_studied"],
    skyline["final_grade_numeric"],
)

print(f"Pearson r:  {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Spearman r: {spearman_r:.3f} (p = {spearman_p:.4f})")

Pearson r:  0.047 (p = 0.6408)
Spearman r: -0.003 (p = 0.9768)


In [18]:
fig = px.scatter(
    skyline,
    x="hours_studied",
    y="final_grade_numeric",
    title=" total hours studied vs final grade",
    trendline="ols",
)
fig.show("browser")

Not plausibly causal, for two reasons: there's essentially no relationship to explain in the first place, and even a stronger correlation couldn't rule out confounds like prior ability, motivation, or course difficulty driving both variables independently.

To claim causation, you'd need a randomized controlled trial

1. Prior studies: students who entered the course with stronger foundational knowledge
2. Motivation: interest in the subject could drive both behaviors independtly

In [21]:
np.random.seed(43)

n = 3060  #(α=0.05, power=0.80, 40%→45%)

engagement = np.random.choice([0, 1], size=n)

# Random assignment to treatment (planner enabled) or control
treatment = np.random.choice([0, 1], size=n)

# B-or-higher propensity depends on engagement, plus a true (modest) planner effect
true_treatment_effect = 0.14  # in z-score units

grade_propensity = (
    engagement
    + true_treatment_effect * treatment
    + np.random.normal(loc=0, scale=1.0, size=n)
)
b_or_higher = (grade_propensity > 0.78).astype(int)  # cutoff tuned so baseline ≈ 40%

ab_test = pd.DataFrame({
    "treatment": treatment,
    "b_or_higher": b_or_higher,
})

# B-or-higher rates by treatment group
ab_results = ab_test.groupby("treatment")["b_or_higher"].agg(["mean", "count"])
ab_results.index = ["Control", "Treatment (Planner)"]
print("A/B test B-or-higher rates:")
print(ab_results)

# Two-proportion z-test for the difference
control_conv = ab_test[ab_test["treatment"] == 0]["b_or_higher"]
treatment_conv = ab_test[ab_test["treatment"] == 1]["b_or_higher"]

n_control = len(control_conv)
n_treatment = len(treatment_conv)
p_control = control_conv.mean()
p_treatment = treatment_conv.mean()

# Pooled proportion for the test
p_pool = (control_conv.sum() + treatment_conv.sum()) / (n_control + n_treatment)
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
z = (p_treatment - p_control) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

# 95% CI for the difference in proportions
se_diff = np.sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)
diff = p_treatment - p_control
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"\nPlanner effect: {diff:.3f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")
print(f"z = {z:.3f}, p = {p_value:.4f}")

A/B test B-or-higher rates:
                         mean  count
Control              0.415576   1528
Treatment (Planner)  0.474543   1532

Planner effect: 0.059
95% CI: (0.024, 0.094)
z = 3.282, p = 0.0010
